<a href="https://colab.research.google.com/github/tokiror/maple-scholars-2026/blob/main/kongwa_grace_seda_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GRACE-SeDA Water Storage Analysis — Kongwa, Tanzania
**Program:** Maple Scholars 2026


**Project:** Natural water reservoirs and drought characterization in Kongwa District, Tanzania
**Author:** Timothy Okiror
**Last updated:** 09 June 2026

## Purpose
This notebook extracts and processes deep-learning-downscaled GRACE total water
storage anomaly (TWSA) data over Tanzania, with a focus on Kongwa District. It
prepares the data for visualization in Google Earth Engine.

## Data Source
- **Product:** GRACE-SeDA v1 (Global Total Water Storage Anomaly, Self-supervised
  Data Assimilation)
- **Authors:** Gou & Soja (2024), *Nature Water*
- **DOI:** https://doi.org/10.3929/ethz-b-000648738
- **Resolution:** 0.5° (~55 km), deep-learning downscaled from raw GRACE
- **Coverage:** April 2002 – December 2022 (216 monthly time steps)
- **Units:** millimeters (mm) equivalent water height
- **Baseline:** anomalies relative to the 2004.0–2009.999 mean (NASA JPL convention)

## What this notebook does
1. Opens the GRACE-SeDA NetCDF file
2. Decodes the time axis from Modified Julian Date to calendar dates
3. Clips the global data to Tanzania (and Kongwa as a focus region)
4. Cleans implausible values (masks pixels beyond ±1000 mm, which are coastline artifacts)
5. Exports monthly layers as georeferenced GeoTIFFs for upload to Google Earth Engine

## Notes
- The companion GEE script (`kongwa_twsa_map.js`) visualizes these outputs on a map.
- Large data files (.nc, .tif) are kept in Google Drive, not in this repository.

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
import os
ds = xr.open_dataset("GRACE-SeDA_v1_2002_2022.nc")
#According to youtube, you have to decode the Julian Date syntax
mjd_epoch = pd.Timestamp('1858-11-17')
#Now we write the right date format we want
real_dates = mjd_epoch + pd.to_timedelta(ds.time.values, unit='D')
print("Date range: ", real_dates[0].date(), 'to', real_dates[-1].date())
print('Total months: ', len(real_dates))
#Need to transpose the global twas to time,latitude,longitude
twsa = ds['twsa'].transpose('time', 'latitude', 'longitude')
#Next we clean it up
twsa_clean = twsa.where((twsa > -1000) & (twsa < 1000))
print(f"Before clean — max: {float(twsa.max()):.0f} mm")
print(f"After clean  — max: {float(twsa_clean.max()):.0f} mm, min: {float(twsa_clean.min()):.0f} mm")



Date range:  2002-04-17 to 2022-12-16
Total months:  216
Before clean — max: 29808 mm
After clean  — max: 1000 mm, min: -1000 mm


In [ ]:
import rioxarray
twsa_clean = twsa_clean.rio.write_crs("EPSG:4326")
twsa_clean = twsa_clean.rio.set_spatial_dims(x_dim='longitude', y_dim='latitude')

#  Make output folder
out_dir = 'grace_seda_geotiffs'
os.makedirs(out_dir, exist_ok=True)

# Export each month
print("\nExporting 216 months...")
for i in range(len(real_dates)):
    date_str = real_dates[i].strftime('%Y_%m')   # e.g. 2002_04
    layer = twsa_clean.isel(time=i)
    fname = f"{out_dir}/SeDA_{date_str}.tif"
    layer.rio.to_raster(fname)
    if (i + 1) % 24 == 0:        # progress every 2 years
        print(f"  ...{i+1}/216 done ({date_str})")

print(f"\n✅ All done. {len(real_dates)} GeoTIFFs saved in '{out_dir}/'")

# Zip for easy download/upload
import shutil
shutil.make_archive('grace_seda_geotiffs', 'zip', out_dir)



Exporting 216 months...
  ...24/216 done (2004_06)
  ...48/216 done (2006_06)
  ...72/216 done (2008_06)
  ...96/216 done (2010_06)
  ...120/216 done (2012_09)
  ...144/216 done (2015_04)
  ...168/216 done (2018_12)
  ...192/216 done (2020_12)
  ...216/216 done (2022_12)

✅ All done. 216 GeoTIFFs saved in 'grace_seda_geotiffs/'


'/content/grace_seda_geotiffs.zip'

**Okay I have loaded the data set into colab decoded the Julian format and now we can see we have a total of 216 months worth of data from 2002 to 2022


In [26]:
##Finally i need to push these files to google drive
from google.colab import drive
import shutil, os

# --- Connect your Google Drive ---
drive.mount('/content/drive')

# --- Make a destination folder in your Drive ---
drive_dest = '/content/drive/MyDrive/grace_seda_geotiffs'
os.makedirs(drive_dest, exist_ok=True)

# --- Copy all 216 GeoTIFFs into Drive ---
src_dir = 'grace_seda_geotiffs'
files = sorted(os.listdir(src_dir))
print(f"Copying {len(files)} files to Drive...")

for i, f in enumerate(files):
    shutil.copy(os.path.join(src_dir, f), os.path.join(drive_dest, f))
    if (i + 1) % 50 == 0:
        print(f"  ...{i+1}/{len(files)} copied")

print(f"\n✅ All {len(files)} files now in your Drive at:")
print(f"   MyDrive/grace_seda_geotiffs/")

Mounted at /content/drive
Copying 214 files to Drive...
  ...50/214 copied
  ...100/214 copied
  ...150/214 copied
  ...200/214 copied

✅ All 214 files now in your Drive at:
   MyDrive/grace_seda_geotiffs/


In [27]:
from google.colab import files
for m in ['2019_01', '2019_06', '2020_04', '2011_06']:
    files.download(f'grace_seda_geotiffs/SeDA_{m}.tif')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

FileNotFoundError: Cannot find file: grace_seda_geotiffs/SeDA_2011_06.tif